# 探索4,119対局：追加10特徴量
段位16以下/18以上、17除外、初期1500・同一時間帯・共通対局除外。探索末2024-07-01 02:00固定。追加8割合＋2量的。モデル学習なし。A1→A2→A3→A4→A5→A6。A4で親照合成功を確認してA5へ進む。

In [1]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os,sys,json,subprocess,urllib.request,hashlib
COMMIT='860605cf0f35f376be9dbd79763775c4300f11b4'
BASE=f'https://raw.githubusercontent.com/yuukienomoto/report_-syobon_n/{COMMIT}/research_v2/fast_release_20261126'
HOME=Path('/tmp')/('received_fast_'+COMMIT[:12])
SOURCE=HOME/'snapshot/research_v2'
manifest=json.loads(urllib.request.urlopen(BASE+'/source_manifest.json').read())
for relative,expected in manifest.items():
    p=HOME/'snapshot'/relative;p.parent.mkdir(parents=True,exist_ok=True)
    content=urllib.request.urlopen(BASE+'/snapshot/'+relative).read()
    if hashlib.sha256(content).hexdigest()!=expected: raise ValueError('固定ソース不一致: '+relative)
    if p.exists() and p.read_bytes()!=content: raise ValueError('ローカル固定版が変更されています: '+relative)
    if not p.exists():p.write_bytes(content)
for name in ['quality_supplement.py','test_supplement.py']:
    (HOME/name).write_bytes(urllib.request.urlopen(BASE+'/'+name).read())
DEPS=HOME/'deps';stamp=DEPS/'installed'
if not stamp.exists():
    subprocess.check_call([sys.executable,'-m','pip','install','-q','--upgrade','--target',str(DEPS),'-r',str(SOURCE/'features_100/requirements.txt')])
    stamp.write_text(COMMIT)
ENV=dict(os.environ,PYTHONPATH=os.pathsep.join(map(str,[DEPS,HOME,SOURCE,SOURCE/'choices',SOURCE/'smoothing'])),PYTHONIOENCODING='utf-8',MPLBACKEND='Agg')
subprocess.check_call([sys.executable,'-m','unittest','discover','-s',str(SOURCE/'features_100/tests'),'-t',str(SOURCE),'-q'],env=ENV)
subprocess.check_call([sys.executable,'-m','unittest','test_supplement','-q'],env=ENV,cwd=HOME)
subprocess.run(['apt-get','install','-y','-qq','fonts-noto-cjk'],check=False,stdout=subprocess.DEVNULL)
print('固定ソース照合完了。実牌譜の検証完了ではありません。')


Mounted at /content/drive
固定ソース照合完了。実牌譜の検証完了ではありません。


## A2 追加モジュール取得と合成テスト
固定高速版は変更しません。Z・全90候補の再計算を避けます。テスト成功は実牌譜検証を意味しません。

In [2]:
ADD_COMMIT='70d6a64952610de562682c173321d45f641ad19d'
ADD=HOME/('additions_'+ADD_COMMIT[:12]);ADD.mkdir(exist_ok=True)
for name in ['core.py','workflow.py','test_additions.py','definitions.csv','README.md']:
    data=urllib.request.urlopen(f'https://raw.githubusercontent.com/yuukienomoto/report_-syobon_n/{ADD_COMMIT}/research_v2/exploration_additions/{name}').read()
    target=ADD/name
    if target.exists() and target.read_bytes()!=data:raise ValueError('固定版不一致: '+name)
    if not target.exists():target.write_bytes(data)
ENV2=dict(ENV,PYTHONPATH=str(ADD)+os.pathsep+ENV['PYTHONPATH'])
subprocess.check_call([sys.executable,'-m','unittest','test_additions','-q'],env=ENV2,cwd=ADD)


0

## A3 対象・入力確認
旧全件カウンタの署名/ハッシュ/IDを照合。4119対局（3467/652）と合わなければ停止します。計数を合わせるために対象を削除しないでください。

In [5]:

ROOT = Path('/content/drive/MyDrive/卒論')

CONFIG = dict(
    run_dir=str(ROOT/'research_v2_runs/20261105T071722Z_5da6c9ad'),
    baseline_dir=str(ROOT/'research_v2_features_100_fast/20261126T100602Z_994332bc'),
    player='しょぼんｎ',
    key='shobon',
    zip_override=None,
    output_root=str(ROOT/'research_v2_exploration_additions')
)

CFG = ADD/'config.json'
CFG.write_text(
    json.dumps(CONFIG, ensure_ascii=False),
    encoding='utf8'
)

def run(mode, extra=()):
    print(f"=== {mode} 開始 ===", flush=True)

    result = subprocess.run(
        [sys.executable, '-u', str(ADD/'workflow.py'),
         mode, str(CFG), *extra],
        env=ENV2,
        cwd=ADD,
        capture_output=True,
        text=True,
        encoding='utf-8'
    )

    print(result.stdout)

    if result.stderr:
        print(result.stderr)

    print("終了コード:", result.returncode)
    result.check_returncode()

run('preview')


=== preview 開始 ===
対象 4119 署名 cc24aaa0809c8a5d519ffc32ea6cdfc309bcff71476f034df293691f5448e3bc

終了コード: 0


## A4 実牌譜パイロット・時間推定
各群3対局。親4項目の全計数/状態/値を既存全件結果と完全比較し不一致で停止。E5と同じIDとは限りません。pilotの新しい日時フォルダのaudit.csv、traces、complete.jsonを確認。未キャッシュ対局時間の4119対局単純外挿を出力。少数ログからの推定であり本番実測ではありません。

In [6]:
run('pilot')


=== pilot 開始 ===
対象 6 探索境界 2024-07-01 02:00:00
1/6 2012062304gm-0029-0000-dff1fa7e cache=True
2/6 2014061300gm-00a9-0000-95909518 cache=True
3/6 2015010523gm-00a9-0000-e2cd35a0 cache=True
4/6 2017110209gm-00a9-0000-6ea2b556 cache=True
5/6 2023042323gm-00a9-0000-e4be9b56 cache=True
6/6 2024063021gm-00a9-0000-7c88ea3a cache=True
保存先: /content/drive/MyDrive/卒論/research_v2_exploration_additions/pilot/20261127T015816Z_94c5a0c4

終了コード: 0


In [7]:

from pathlib import Path
import pandas as pd

pilot_dir = Path(
    "/content/drive/MyDrive/卒論/"
    "research_v2_exploration_additions/"
    "pilot/20261127T015816Z_94c5a0c4"
)

for p in sorted(pilot_dir.rglob("*")):
    if p.is_file():
        print(p.relative_to(pilot_dir))

        if p.suffix in [".json", ".md", ".txt"]:
            if p.stat().st_size < 20000:
                print(p.read_text(encoding="utf-8"))

        elif p.suffix == ".csv":
            df = pd.read_csv(p)
            print("行数:", len(df))
            if len(df) <= 30:
                print(df.to_string(index=False))
            else:
                print("列名:", df.columns.tolist())


audit.csv
行数: 6
                        game_id status  reused  seconds
2012062304gm-0029-0000-dff1fa7e     ok    True 0.008534
2014061300gm-00a9-0000-95909518     ok    True 0.003692
2015010523gm-00a9-0000-e2cd35a0     ok    True 0.003756
2017110209gm-00a9-0000-6ea2b556     ok    True 0.003317
2023042323gm-00a9-0000-e4be9b56     ok    True 0.003742
2024063021gm-00a9-0000-7c88ea3a     ok    True 0.003399
complete.json
{
  "signature": "cc24aaa0809c8a5d519ffc32ea6cdfc309bcff71476f034df293691f5448e3bc",
  "mode": "pilot",
  "rows": 6,
  "sha256": "f9ec9614e204945bc5b29362fde7978bad7c5c7b3a6e8049147c4f828b5f310f",
  "end": "2024-07-01 02:00:00",
  "elapsed": 0.1654284919999327,
  "extrapolated_4119_hours": null,
  "n_uncached": 0,
  "timing_limitation": "同環境の単純外挿。牌譜長・Drive・キャッシュ温まりで変動"
}
counters.csv
行数: 6
                        game_id  dan                hour status  P01__numerator  P01__denominator  P01__other  P01__unknown  P01__excluded  P01__censored    P01__status  P01__value  Q01

## A5 探索期間全件抽出・中断再開
A4の監査を確認後に実行。1セッションだけで実行。成功追加キャッシュを再利用、詳細トレースは保存しません。親カウンタ照合は全4119件で行います。中断後は同じ版・設定でA1〜A3、A5を実行。残存ロックは自動削除しません。

In [9]:
RUN_FULL=True # A4確認後True
if RUN_FULL:run('full')
else:print('パイロット監査を確認してからRUN_FULL=Trueにしてください')


=== full 開始 ===
対象 4119 探索境界 2024-07-01 02:00:00
1/4119 2012062304gm-0029-0000-dff1fa7e cache=True
2/4119 2012062311gm-0029-0000-f8bbd45f cache=False
3/4119 2012062312gm-0029-0000-006c5ee0 cache=False
4/4119 2012062316gm-0029-0000-c3810c0f cache=False
5/4119 2012062317gm-0029-0000-94e13395 cache=False
6/4119 2012062318gm-0029-0000-e26b3cd9 cache=False
7/4119 2012062422gm-0029-0000-5d7681c0 cache=False
8/4119 2012062422gm-0029-0000-5dfa64d8 cache=False
9/4119 2012062423gm-0029-0000-3de5f15c cache=False
10/4119 2012062500gm-0029-0000-fe9a5984 cache=False
11/4119 2012062501gm-0029-0000-84a18c6b cache=False
12/4119 2012062502gm-0029-0000-187105aa cache=False
13/4119 2012062503gm-0029-0000-bee1150b cache=False
14/4119 2012062513gm-0029-0000-2f5ca89e cache=False
15/4119 2012062514gm-0029-0000-ea6d2697 cache=False
16/4119 2012062515gm-0029-0000-2ff20861 cache=False
17/4119 2012062516gm-0029-0000-23257b3a cache=False
18/4119 2012062517gm-0029-0000-25a2a162 cache=False
19/4119 2012062517gm-0029

## A6 既存90＋追加10の品質比較
追加全件完了フォルダを指定。1対局・非重複10/20対局の統計、親子対応、時期別・群別相関。評価期間の数値は集計しません。量的に二項モデルを適用しません。分母0/unknownは欠損。

In [11]:

FULL_DIR = '/content/drive/MyDrive/卒論/research_v2_exploration_additions/full/20261127T020204Z_44a6bd74'

run('quality', ['--folder', FULL_DIR])


=== quality 開始 ===


KeyboardInterrupt: 

## 出力と限界
quality/日時 内のgroups.csv、comparison.csv、parent_child_same_windows.csv、periods.csv、correlations.csv、values.csv、report_ja.md。width=1は対局単位。10/20を混ぜない。群内分散とSMDだけで採否を決めず、機会数・欠損・時期差を確認。局内/時間依存の二項回帰は未実施。全期間の予測性能や因果効果ではありません。